# Noise and finite-shot analysis

Evaluate the analytical circuit-error model and sufficient finite-shot budgets; generates `noise_budget.csv`. The first code cell also provides the model used by the figure notebook.

Run the cells in order using the Python 3 kernel in the repository root. Dependencies are listed in `requirements.txt`.

## Pauli propagation, witness scores and shot bounds

In [ ]:
"""Analytic Pauli propagation for a five-CZ preparation, plus shot budgets."""
import csv,math
from scipy.optimize import brentq
edges=[(0,1),(2,3),(1,2),(3,4),(4,0)]
strings=['XZIIZ','ZXZII','IZXZI','IIZXZ','ZIIZX','XXYIY','XXXXX','XYZZY','XYIYX','YXYZZ','YXXYI','YYZIZ','YYIXI']
def info(s):
 x=[int(c in 'XY') for c in s];z=[int(c in 'ZY') for c in s];d=0
 for a,b in reversed(edges):
  d+=bool(x[a] or z[a] or x[b] or z[b]);z[a]^=x[b];z[b]^=x[a]
 assert not any(z),s
 return d,sum(c!='I' for c in s)
params=[info(s) for s in strings]
b=2*math.sqrt(2);lam=4-b
lam_star=b-2
bounds=[2+math.sqrt(5),b,b,2];groups=[[2,2,1],[1,1,1,1+lam],[.5]*6+[1],[.5]*6+[1+lam_star]]
weights=list(map(sum,groups))
def scores(q,r,v=1):
 p=[v*(1-q)**d*(1-2*r)**w for d,w in params];return [sum(p[:5]),sum(p[5::2])-lam*(1-p[3]),sum(p[5:])/2,sum(p[5:])/2-lam_star*p[3]]
# p[5::2] indices 5 7 9 11

def shots(s,B,gs,delta=.01):
 return sum(math.ceil(8*math.log(1/delta)*sum(gs)*g/(s-B)**2) for g in gs) if s>B else None


## Evaluate thresholds and export budgets

In [ ]:
if __name__=='__main__':
 for q,r,v in [(0,0,.9),(.01,.01,1),(.02,.02,1),(.03,.02,1)]:
  ss=scores(q,r,v);print(q,r,v,ss,[shots(s,B,L) for s,B,L in zip(ss,bounds,groups)])
 print('gate thresholds at r=.01',[brentq(lambda q:scores(q,.01)[i]-bounds[i],0,.2) for i in range(4)])
 with open('noise_budget.csv','w') as f:
  w=csv.writer(f);w.writerow(['gate_q','readout_r','generator','penalty','projected','visibility','generator_shots','penalty_shots','projected_shots','visibility_shots']);
  for q in [i/1000 for i in range(101)]:
   ss=scores(q,.01);w.writerow([q,.01,*ss,*[shots(s,B,L) for s,B,L in zip(ss,bounds,groups)]])


## Regression checks against the revised manuscript
The reported budgets are sufficient worst-case Hoeffding budgets, not optimized sample complexities. The gate/readout and white-noise scores are model-conditional.

In [ ]:
# Regression checks against the revised manuscript's matched-budget table.
expected_global = [13223, 2302, 992, 1178]
expected_circuit = [8448, 3153, 1295, 1506]
for (q, r, v), expected in [((0, 0, .90), expected_global), ((.01, .01, 1), expected_circuit)]:
    computed = [shots(s, B, gs) for s, B, gs in zip(scores(q, r, v), bounds, groups)]
    assert computed == expected, (q, r, v, computed, expected)

# Model-conditional gate thresholds at independent 1% readout errors.
gate_endpoints = [brentq(lambda q: scores(q, .01)[i] - bounds[i], 0, .2) for i in range(4)]
expected_gate_endpoints = [.034536637761, .040543601548, .054983527646, .074297217444]
assert all(abs(a-b) < 1e-9 for a,b in zip(gate_endpoints, expected_gate_endpoints))

# White-noise tolerances: the last is an exact fixed-frame assemblage
# threshold; the others are thresholds for their respective scores.
white_noise = [1 - bounds[0]/5, 1 - (bounds[1]+lam)/(4+lam),
               1 - bounds[2]/4, 1 - 2/(4-lam_star)]
assert all(abs(a-b)<1e-5 for a,b in zip(white_noise,[.1527864,.2265409,.2928932,.3693981]))
print("Four-witness white-noise tolerances (%):", [round(100*x,4) for x in white_noise])
print("Manuscript shot budgets and gate thresholds: PASS")
